# 19 - Training and registry
Run the setup cell first, then the work cells in order. Use the WSL mlop-pj kernel.
These cells change cloud resources when you run them. A pending result needs a wait and a phase retry, not a new job.
Use notebooks 18-22 for the five main parts. Training and log waits can overlap with GitHub setup. See docs/RUN_FINISH.md.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert Path(sys.prefix).name == "mlop-pj", "Select the WSL mlop-pj kernel."
assert "microsoft" in os.uname().release.lower(), "Open this project in WSL."
status = json.loads((ROOT / "reports/azure-blob-upload.json").read_text())
assert status.get("data_uploaded") and status.get("all_hashes_match") and not status["errors"], "Finish 17 first."

def step(phase):
    print("Running:", phase, flush=True)
    process = subprocess.run([sys.executable, "-m", "cloudlayer.finish", phase],
                             cwd=ROOT, capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError("Phase process failed. Check its stderr before retrying: " + process.stderr)
    result = json.loads(process.stdout)
    print(json.dumps(result, indent=2))
    if result["errors"]:
        raise RuntimeError(f"Check reports/finish-{phase}.json and the matching log in tmp. Stop this group.")
    if result.get("pending"):
        print("Still waiting. Read next in the report; do not recreate the resource or job.")
    return result

print("Root:", ROOT)
print("17 passed. Run this notebook work cells in order.")

Root: /mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ
17 passed. Run this notebook work cells in order.


## Group 2 — Managed training and registry
Submit one Azure ML job from the versioned profiles. It fits the selected pipeline again, checks numerical equivalence, and exports the tested serving bundle with a training proof. This keeps the image and registry artifact tied to the same bundle hash.

The compute has min=0, max=1, and an idle shutdown setting. `training-submit` saves the job name. `training-registry` polls that job and registers its output only after it completes and the proof passes. If it is pending, move to Group 3, then rerun only the registry cell. Do not submit a second job.

Outputs: `finish-training-submit.json`, `finish-training-registry.json`, and `azure-training-proof.json`. Final check: `registry_ready: true`.

In [2]:
training = step("training-submit")
registry = step("training-registry")

Running: training-submit
{
  "checked_at": "2026-10-11T16:41:50.894220+00:00",
  "phase": "training-submit",
  "errors": [],
  "job_name": "train-20261011-164656",
  "image_uri": "acrmloppj492ea24d.azurecr.io/travel-style@sha256:05d1819321f472b39fe1035cb9cee68fc7432a9ad2be4b56afdc8c85af562fcc",
  "submitted": true,
  "status": "Starting"
}
Running: training-registry
{
  "checked_at": "2026-10-11T16:49:30.789318+00:00",
  "phase": "training-registry",
  "errors": [],
  "job_name": "train-20261011-164656",
  "status": "Queued",
  "registry_ready": false,
  "pending": true,
  "next": "Training is still running. Continue independent cells, then rerun this cell."
}
Still waiting. Read next in the report; do not recreate the resource or job.
